# 03. Sub Byte Bitpacking, T-MAC Table Lookup GEMM, and GGUF K-Quants

- **Category:** 14_quantisation / 02_pt2e_torchao_and_bitpacking
- **Lineage & Citations:** Gerganov & Kawrakow (2023, *llama.cpp GGUF k-quants & imatrix architecture*); Wei et al. (2024, *T-MAC: CPU Renaissance via Table Lookup for Low-Bit LLM Deployment*).
- **Purpose:** Formulate physical sub-byte bitpacking ($2\times\text{INT4}$, $4\times\text{INT2}$, $5\times\text{Ternary}$ into bytes), T-MAC bit-serial lookup table matrix multiplication without floating-point multiplies, and GGUF `Q4_K_M` hierarchical super-block quantization.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. TorchAO Modern API and Affine Quantized Tensor](02_torchao_modern_api_and_affine_quantized_tensor.ipynb) $\to$ `🔗 Downstream Syntheses:` [01. RTN vs GPTQ Hessian Error Compensation](../03_llm_ptq_algorithms/01_rtn_vs_gptq_hessian_error_compensation.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT store 4-bit, 2-bit, or ternary weights inside `torch.int8` tensors without bitpacking (saves **0% memory** in PyTorch!); do NOT expand T-MAC LUT group sizes beyond $g=4$ ($2^8 = 256$ entries exceeds CPU vector registers, causing L1 cache stalls).

## Mathematical Foundations & Sub-Byte Packing Mechanics

1. **$2 \times \text{INT4}$ Packing into Byte**:
   $$\text{packed} = (v_0 \& 0x0F) \mid ((v_1 \& 0x0F) \ll 4) \implies 50\% \text{ memory reduction}$$
2. **$5 \times \text{Ternary}$ Packing ($3^5 = 243 < 256$)**:
   $$\text{code} = t_0 + 3 t_1 + 9 t_2 + 27 t_3 + 81 t_4 \implies 1.58 \text{ bits/weight packed density}$$
3. **T-MAC 16-Entry Precomputed Activation Table**:
   $$\text{LUT}[i] = \sum_{k=0}^3 ((i \gg k) \& 1) \cdot a_k, \quad \mathbf{w} \cdot \mathbf{a} = \sum_{p=0}^{b-1} 2^p \cdot \text{LUT}[\text{index}(\mathbf{w}^{(p)})]$$
4. **GGUF `Q4_K_M` Super-Block Layout**:
   $$N=256 \text{ weights} = 128\text{B (packed nibbles)} + 8\text{B (sub-scales)} + 2\text{B (super-scale)} \implies 4.31 \text{ bits/param}.$$

In [ ]:
import sys, time, math
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/14_quantisation/02_pt2e_torchao_and_bitpacking/03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def linear_anchor(x, weight, bias):
    return x @ weight + bias

def bitpack_2x4bit(v_int4):
    v = v_int4.long()
    return ((v[0::2] & 0x0F) | ((v[1::2] & 0x0F) << 4)).float()

def bitunpack_2x4bit(packed):
    p = packed.long()
    out = torch.empty(len(p) * 2, dtype=torch.float32, device=packed.device)
    out[0::2], out[1::2] = (p & 0x0F).float(), ((p >> 4) & 0x0F).float()
    return out

def bitpack_5x_ternary(trits):
    t = trits.long().reshape(-1, 5)
    return (t[:, 0] + 3 * t[:, 1] + 9 * t[:, 2] + 27 * t[:, 3] + 81 * t[:, 4]).float()

def bitunpack_5x_ternary(packed_codes):
    c = packed_codes.long()
    t0 = c % 3; c1 = c // 3; t1 = c1 % 3; c2 = c1 // 3; t2 = c2 % 3; c3 = c2 // 3; t3 = c3 % 3; t4 = c3 // 3
    return torch.stack([t0, t1, t2, t3, t4], dim=-1).flatten().float()

def tmac_precompute_lut(activations_group4):
    lut = torch.zeros(16, dtype=torch.float32, device=activations_group4.device)
    a = activations_group4.flatten()
    for i in range(16):
        lut[i] = sum(float(a[k]) for k in range(4) if (i >> k) & 1)
    return lut

def tmac_lut_lookup(lut, weight_indices):
    return lut[weight_indices.long()]

def gguf_q4_k_quantize_block(w_block256):
    w = w_block256.flatten()
    d = max(float(torch.max(torch.abs(w))) / 127.0, 1e-8)
    sub_scales, dequants = [], []
    for k in range(8):
        chunk = w[k * 32 : (k + 1) * 32]
        cmax = float(torch.max(torch.abs(chunk)))
        dk = cmax / (7.0 * d) if cmax > 0.0 else 1.0
        eff_scale = d * dk
        dequants.append(torch.clamp(torch.round(chunk / eff_scale), -8.0, 7.0) * eff_scale)
        sub_scales.append(dk)
    return torch.cat(dequants), torch.tensor([d]), torch.tensor(sub_scales)

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert torch.max(torch.abs(y_anchor - weights["test_output"])) < 1e-6

# 2. 2x4-bit packing parity
p2x4 = bitpack_2x4bit(weights["raw_nibbles"])
assert torch.max(torch.abs(p2x4 - weights["packed_2x4bit"])) < 1e-5
unp2x4 = bitunpack_2x4bit(weights["packed_2x4bit"])
assert torch.max(torch.abs(unp2x4 - weights["unpacked_nibbles"])) < 1e-5

# 3. 5x ternary packing parity
p_trit = bitpack_5x_ternary(weights["raw_trits"])
assert torch.max(torch.abs(p_trit - weights["packed_ternary"])) < 1e-5
unp_trit = bitunpack_5x_ternary(weights["packed_ternary"])
assert torch.max(torch.abs(unp_trit - weights["unpacked_trits"])) < 1e-5

# 4. T-MAC LUT parity
lut = tmac_precompute_lut(weights["tmac_act"])
assert torch.max(torch.abs(lut - weights["tmac_lut"])) < 1e-5
tmac_out = tmac_lut_lookup(lut, weights["tmac_weight_indices"])
assert torch.max(torch.abs(tmac_out - weights["tmac_dot_ref"])) < 1e-5

# 5. GGUF Q4_K parity
deq, sup_s, sub_s = gguf_q4_k_quantize_block(weights["gguf_raw_w"])
assert torch.max(torch.abs(deq - weights["gguf_dequant"])) < 1e-5
assert torch.max(torch.abs(sup_s - weights["gguf_super_scale"])) < 1e-5
assert torch.max(torch.abs(sub_s - weights["gguf_sub_scales"])) < 1e-5
print("All Bitpacking, T-MAC LUT, and GGUF Q4_K parity assertions verified!")

In [ ]:
# Microbenchmark: T-MAC Table Lookup vs FP32 Multiplier Dot Product
indices_bench = torch.randint(0, 16, (1000,))
t0 = time.perf_counter()
for _ in range(100): _ = tmac_lut_lookup(weights["tmac_lut"], indices_bench)
t_lut = (time.perf_counter() - t0) / 100 * 1e6

w_float = weights["tmac_weight_indices"].repeat(250)
a_float = weights["tmac_act"].repeat(250)
t0 = time.perf_counter()
for _ in range(100): _ = w_float * a_float
t_fp = (time.perf_counter() - t0) / 100 * 1e6

print(f"T-MAC Table Lookup latency: {t_lut:.2f} us (Zero float multiplications)")
print(f"Standard Float MAC latency:  {t_fp:.2f} us")

In [ ]:
# Real-World RAM Savings: Unpacked vs Bitpacked Storage Footprints
n_weights = 100_000
# 1. Unpacked PyTorch int8
unpacked_bytes = n_weights * 1  # 1 byte per element in torch.int8

# 2. Bitpacked 4-bit (2 elements per byte)
packed_4bit_bytes = n_weights // 2

# 3. Bitpacked Ternary (5 elements per byte)
packed_ternary_bytes = n_weights // 5

print(f"100K weights raw int8 storage:     {unpacked_bytes / 1024:.1f} KB (0% RAM saved)")
print(f"100K weights bitpacked 4-bit:      {packed_4bit_bytes / 1024:.1f} KB (50.0% RAM saved!)")
print(f"100K weights bitpacked Ternary:    {packed_ternary_bytes / 1024:.1f} KB (80.0% RAM saved!)")
reduction_4bit = unpacked_bytes / packed_4bit_bytes
reduction_ternary = unpacked_bytes / packed_ternary_bytes
assert reduction_4bit == 2.0, "4-bit packing must yield exactly 2x memory reduction!"
assert reduction_ternary == 5.0, "Ternary packing must yield exactly 5x memory reduction!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Physical Sub-Byte Densification**: Bitpacking maps fractional-byte structures into integer word units, yielding genuine $2\times$ (4-bit) and $5\times$ (ternary) hardware RAM reductions.
2. **T-MAC Multiplier Elimination**: Precomputing 16-entry activation sums reduces low-bit GEMM to table lookups and additions, running up to $4\times$ faster on edge CPUs than dequantize-to-FP16.
3. **GGUF Hierarchical Scaling**: Super-block grouping amortizes scale metadata to $0.31\text{ bits/param}$, preventing dynamic range clipping while keeping weights tightly packed.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.md](../../../markdown/14_quantisation/02_pt2e_torchao_and_bitpacking/03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.md)
- **PyTorch 2.x Module:** [03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.py](../../../pytorch2/14_quantisation/02_pt2e_torchao_and_bitpacking/03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.py)
- **Functional JAX Module:** [03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.py](../../../jax/14_quantisation/02_pt2e_torchao_and_bitpacking/03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.py)
- **Native Rust Candle Track:** [03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.rs](../../../candle/14_quantisation/02_pt2e_torchao_and_bitpacking/03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.rs)
- **Hardware Kernel (CPU / WGSL):** [03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.rs](../../../kernels/14_quantisation/02_pt2e_torchao_and_bitpacking/03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.rs)
- **Golden SafeTensors:** [03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.safetensors](../../../weights/14_quantisation/02_pt2e_torchao_and_bitpacking/03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. TorchAO Modern API and Affine Quantized Tensor](02_torchao_modern_api_and_affine_quantized_tensor.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 01. RTN vs GPTQ Hessian Error Compensation](../03_llm_ptq_algorithms/01_rtn_vs_gptq_hessian_error_compensation.ipynb) &rarr;